# Topic 9a: Linear Classification, and How to Judge a Classifier

CSCI 3320 · AI Programming in Python · Fall 2026

This notebook is the code from the [Topic 9a lecture note](https://dongchul.kim/csci3320-fall2026/topic/9), every cell in the
same order, under the note's own section headings. The explanations are in the note: keep it
open beside this notebook. Run the cells from top to bottom (Runtime > Run all works too),
then change things and run them again.

It runs on a CPU; no GPU is needed. Numbers can differ a little from the note on a
different library version.

## The data: twelve students

In [ ]:
import numpy as np

x = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0])  # hours
y = np.array([  0,   0,   0,   0,   1,   0,   1,   1,   1,   1,   1,   1])    # 1 = passed

for h, passed in zip(x, y):
    print(f"{h:4.1f} hours  ->  {'PASS' if passed else 'fail'}")

print(f"\n{y.sum()} of {len(y)} passed")

## The sigmoid

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

for z in [-6, -2, -1, 0, 1, 2, 6]:
    print(f"sigmoid({z:3}) = {sigmoid(z):.4f}")

## The cost: how wrong is a probability?

In [ ]:
import numpy as np

for p in [0.99, 0.9, 0.5, 0.1, 0.01]:
    print(f"p = {p:5.2f}   if truth is 1: {-np.log(p):6.3f}   if truth is 0: {-np.log(1-p):6.3f}")

## The training loop

In [ ]:
import numpy as np

x = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0])
y = np.array([  0,   0,   0,   0,   1,   0,   1,   1,   1,   1,   1,   1])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

w, b = 0.0, 0.0
alpha = 0.5

print(" step        w         b     cost   accuracy")
for step in range(4001):
    p = sigmoid(w * x + b)                    # forward: probabilities

    if step % 500 == 0:
        cost = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
        acc  = ((p >= 0.5).astype(int) == y).mean()
        print(f"{step:5d}  {w:7.4f}  {b:8.4f}   {cost:.4f}     {acc:.4f}")

    grad_w = np.mean((p - y) * x)             # the gradient, both lines
    grad_b = np.mean(p - y)
    w -= alpha * grad_w                       # the update, both lines
    b -= alpha * grad_b

print(f"\nfinal:  w = {w:.4f}   b = {b:.4f}")
print(f"decision boundary at x = -b/w = {-b/w:.4f} hours")

## What the model learned

In [ ]:
import numpy as np

x = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0])
y = np.array([  0,   0,   0,   0,   1,   0,   1,   1,   1,   1,   1,   1])

w, b = 2.6101, -7.1763                        # what the loop above found
p = 1 / (1 + np.exp(-(w * x + b)))
pred = (p >= 0.5).astype(int)

print("hours   P(pass)  predicted  actual")
for h, prob, pr, ac in zip(x, p, pred, y):
    mark = "   <-- wrong" if pr != ac else ""
    print(f"{h:5.1f}    {prob:.3f}       {pr}         {ac}{mark}")

print(f"\naccuracy = {(pred == y).mean():.4f}  ({(pred == y).sum()} of {len(y)})")

## scikit-learn does all of it

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

x = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0])
y = np.array([  0,   0,   0,   0,   1,   0,   1,   1,   1,   1,   1,   1])

X = x.reshape(-1, 1)                          # scikit-learn wants a 2-D X

model = LogisticRegression(C=1e6)             # C is huge: almost no regularisation
model.fit(X, y)                               # the entire loop above

print(f"scikit-learn  w = {model.coef_[0][0]:.4f}   b = {model.intercept_[0]:.4f}")
print(f"our loop      w = 2.6101   b = -7.1763")
print(f"\nboundary: {-model.intercept_[0]/model.coef_[0][0]:.4f} hours")
print(f"accuracy: {model.score(X, y):.4f}")
print(f"P(pass | 3.5 hours) = {model.predict_proba([[3.5]])[0][1]:.4f}")

## A test that is 98% accurate and completely useless

In [ ]:
import numpy as np

n, n_positive = 1000, 20
truth = np.zeros(n, dtype=int)
truth[:n_positive] = 1

def useless_model(person):
    return 0                                  # "you are healthy". always.

pred = np.array([useless_model(i) for i in range(n)])

print(f"people screened     : {n}")
print(f"actually sick       : {truth.sum()}")
print(f"model's accuracy    : {(pred == truth).mean():.4f}")
print(f"sick people it found: {((pred == 1) & (truth == 1)).sum()}")

## A real model, and four numbers

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
n, n_positive = 1000, 20
truth = np.zeros(n, dtype=int); truth[:n_positive] = 1
rng.shuffle(truth)

# a model that is genuinely informative: sick people score higher, but the
# two groups overlap, which is what makes this interesting
score = np.where(truth == 1, rng.normal(0.72, 0.16, n), rng.normal(0.30, 0.16, n))
score = np.clip(score, 0.001, 0.999)
pred  = (score >= 0.5).astype(int)

TP = ((pred == 1) & (truth == 1)).sum()
FP = ((pred == 1) & (truth == 0)).sum()
FN = ((pred == 0) & (truth == 1)).sum()
TN = ((pred == 0) & (truth == 0)).sum()

print(f"TP = {TP:4d}   FN = {FN:4d}")
print(f"FP = {FP:4d}   TN = {TN:4d}")
print(f"\naccuracy = {(TP + TN) / n:.4f}")
print(f"the useless model got 0.9800")

np.save("/tmp/truth.npy", truth); np.save("/tmp/score.npy", score)   # reused below

## Precision and recall

In [ ]:
import numpy as np

truth = np.load("/tmp/truth.npy"); score = np.load("/tmp/score.npy")
pred = (score >= 0.5).astype(int)

TP = ((pred == 1) & (truth == 1)).sum()
FP = ((pred == 1) & (truth == 0)).sum()
FN = ((pred == 0) & (truth == 1)).sum()
TN = ((pred == 0) & (truth == 0)).sum()

precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1        = 2 * precision * recall / (precision + recall)

print(f"accuracy  = {(TP + TN) / len(truth):.4f}   <- flattered by the 980 healthy people")
print(f"precision = {precision:.4f}   <- of {TP + FP} alarms, {TP} were real")
print(f"recall    = {recall:.4f}   <- of {TP + FN} sick people, it found {TP}")
print(f"F1        = {f1:.4f}")

## F1: one number again, but an honest one

In [ ]:
def f1(p, r):
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)

print(f"{'precision':>10} {'recall':>8} {'mean':>8} {'F1':>8}")
for p, r in [(1.0, 0.0), (0.9, 0.1), (0.5, 0.5), (0.6, 0.4), (0.9, 0.9)]:
    print(f"{p:10.2f} {r:8.2f} {(p+r)/2:8.2f} {f1(p, r):8.2f}")

## The threshold is a dial, not a law

In [ ]:
import numpy as np

truth = np.load("/tmp/truth.npy"); score = np.load("/tmp/score.npy")

print(f"{'threshold':>10} {'TP':>4} {'FP':>5} {'FN':>4} {'precision':>10} {'recall':>8}")
for t in [0.1, 0.3, 0.5, 0.7, 0.9]:
    pred = (score >= t).astype(int)
    TP = ((pred == 1) & (truth == 1)).sum()
    FP = ((pred == 1) & (truth == 0)).sum()
    FN = ((pred == 0) & (truth == 1)).sum()
    prec = TP / (TP + FP) if TP + FP else float("nan")
    print(f"{t:10.1f} {TP:4d} {FP:5d} {FN:4d} {prec:10.3f} {TP/(TP+FN):8.3f}")

## So how do you actually choose the threshold?

In [ ]:
import numpy as np

truth = np.load("/tmp/truth.npy"); score = np.load("/tmp/score.npy")

def at(t):
    pred = (score >= t).astype(int)
    TP = ((pred == 1) & (truth == 1)).sum(); FP = ((pred == 1) & (truth == 0)).sum()
    FN = ((pred == 0) & (truth == 1)).sum(); TN = ((pred == 0) & (truth == 0)).sum()
    prec = TP / (TP + FP) if TP + FP else 0.0
    rec  = TP / (TP + FN)
    f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return TP, FP, FN, TN, prec, rec, f1

grid = np.arange(0.01, 1.00, 0.01)

best_f1 = max(grid, key=lambda t: at(t)[6])                      # 3. maximise F1
best_j  = max(grid, key=lambda t: at(t)[5] - at(t)[1]/(at(t)[1]+at(t)[3]))   # 3. maximise TPR-FPR
cheap   = min(grid, key=lambda t: 20 * at(t)[2] + at(t)[1])      # 1. a miss costs 20 alarms
need95  = max(t for t in grid if at(t)[5] >= 0.95)               # 2. recall must reach 0.95

print(f"{'how you chose':<28}{'t':>6}{'TP':>5}{'FP':>6}{'FN':>5}{'precision':>11}{'recall':>8}")
for name, t in [("the default, 0.5", 0.5),
                ("maximise F1", best_f1),
                ("maximise TPR - FPR", best_j),
                ("a miss costs 20 alarms", cheap),
                ("recall must reach 0.95", need95)]:
    TP, FP, FN, TN, prec, rec, f1 = at(t)
    print(f"{name:<28}{t:>6.2f}{TP:>5}{FP:>6}{FN:>5}{prec:>11.3f}{rec:>8.3f}")

## scikit-learn does all of the evaluation too

In [ ]:
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

truth = np.load("/tmp/truth.npy"); score = np.load("/tmp/score.npy")
pred  = (score >= 0.5).astype(int)

print("confusion matrix, rows = truth, columns = prediction")
print(confusion_matrix(truth, pred))
print("        pred 0   pred 1")
print(" real 0   TN       FP")
print(" real 1   FN       TP")

print(f"\naccuracy  {accuracy_score(truth, pred):.4f}")
print(f"precision {precision_score(truth, pred):.4f}")
print(f"recall    {recall_score(truth, pred):.4f}")
print(f"f1        {f1_score(truth, pred):.4f}")
print(f"roc auc   {roc_auc_score(truth, score):.4f}   <- the SCORES, not the predictions")

print("\n" + classification_report(truth, pred, target_names=["healthy", "sick"]))

## Putting both halves together

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

x = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0])
y = np.array([  0,   0,   0,   0,   1,   0,   1,   1,   1,   1,   1,   1])
X = x.reshape(-1, 1)

model = LogisticRegression(C=1e6).fit(X, y)
pred  = model.predict(X)

print(confusion_matrix(y, pred), "\n")
print(classification_report(y, pred, target_names=["fail", "pass"]))

Back to the [Topic 9a lecture note](https://dongchul.kim/csci3320-fall2026/topic/9).